In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
po_df = spark.read.table("dbt_adv_pr.silver.po_2025_all")
sup_df = spark.read.table("dbt_adv_pr.silver.supplier")
profile_df = spark.read.table("dbt_adv_pr.silver.supplier_profile")

# 1. Observed PO metrics (filter to completed POs where received_date is not null)
po_metrics = (
    po_df
    .filter(F.col("received_date").isNotNull())
    .groupBy("supplier_id")
    .agg(
        F.count("po_id").alias("total_pos"),
        F.avg(
            F.when(F.col("received_date") <= F.col("expected_date"), 1.0).otherwise(0.0)
        ).alias("on_time_rate"),
        F.avg("unit_price").alias("avg_price"),
        F.avg(F.datediff(F.col("received_date"), F.col("order_date"))).alias("avg_lead_time")
    )
)

# 2. QC rating from supplier table (averaged across SKUs per supplier)
qc_metrics = (
    sup_df
    .groupBy("supplier_id")
    .agg(F.avg("qc_rating").alias("avg_qc_rating"))
)

# 3. Combine metrics with supplier_profile for clean supplier names
combined_df = (
    po_metrics
    .join(qc_metrics, on="supplier_id", how="left")
    .join(profile_df.select("supplier_id", "name", "city"), on="supplier_id", how="left")
)

# 4. Min-Max Normalize all 4 metrics onto a 0-100 scale
w_all = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

norm_df = (
    combined_df
    .withColumn("min_qc", F.min("avg_qc_rating").over(w_all))
    .withColumn("max_qc", F.max("avg_qc_rating").over(w_all))
    .withColumn("min_otd", F.min("on_time_rate").over(w_all))
    .withColumn("max_otd", F.max("on_time_rate").over(w_all))
    .withColumn("min_price", F.min("avg_price").over(w_all))
    .withColumn("max_price", F.max("avg_price").over(w_all))
    .withColumn("min_lt", F.min("avg_lead_time").over(w_all))
    .withColumn("max_lt", F.max("avg_lead_time").over(w_all))
    # Higher is better: QC Rating & On-Time Rate
    .withColumn(
        "qc_score",
        F.when(F.col("max_qc") == F.col("min_qc"), F.lit(100.0))
         .otherwise(((F.col("avg_qc_rating") - F.col("min_qc")) / (F.col("max_qc") - F.col("min_qc"))) * 100)
    )
    .withColumn(
        "otd_score",
        F.when(F.col("max_otd") == F.col("min_otd"), F.lit(100.0))
         .otherwise(((F.col("on_time_rate") - F.col("min_otd")) / (F.col("max_otd") - F.col("min_otd"))) * 100)
    )
    # Lower is better: Price & Lead Time
    .withColumn(
        "price_score",
        F.when(F.col("max_price") == F.col("min_price"), F.lit(100.0))
         .otherwise(((F.col("max_price") - F.col("avg_price")) / (F.col("max_price") - F.col("min_price"))) * 100)
    )
    .withColumn(
        "lt_score",
        F.when(F.col("max_lt") == F.col("min_lt"), F.lit(100.0))
         .otherwise(((F.col("max_lt") - F.col("avg_lead_time")) / (F.col("max_lt") - F.col("min_lt"))) * 100)
    )
)

# 5. Apply weights (30% QC, 30% On-Time, 25% Price, 15% Lead Time) and rank
w_qc, w_otd, w_price, w_lt = 0.30, 0.30, 0.25, 0.15

final_scores = (
    norm_df
    .withColumn(
        "composite_score",
        F.round(
            (F.col("qc_score") * F.lit(w_qc)) +
            (F.col("otd_score") * F.lit(w_otd)) +
            (F.col("price_score") * F.lit(w_price)) +
            (F.col("lt_score") * F.lit(w_lt)),
            2
        )
    )
    .filter(F.col("composite_score").isNotNull())
    .withColumn("rank", F.dense_rank().over(Window.orderBy(F.col("composite_score").desc())))
    .select(
        "rank",
        "supplier_id",
        "name",
        "city",
        "total_pos",
        F.round("avg_qc_rating", 2).alias("avg_qc_rating"),
        F.round(F.col("on_time_rate") * 100, 1).alias("on_time_pct"),
        F.round("avg_price", 2).alias("avg_price"),
        F.round("avg_lead_time", 1).alias("avg_lead_time_days"),
        "composite_score"
    )
    .orderBy("rank")
)

final_scores.show(truncate=False)

In [ ]:
final_scores.write.mode('append')\
                  .saveAsTable('dbt_adv_pr.gold.supplier_score')  

In [ ]:
sup_df = spark.read.table("dbt_adv_pr.silver.supplier")
po_df = spark.read.table("dbt_adv_pr.silver.po_2025_all")
profile_df = spark.read.table("dbt_adv_pr.silver.supplier_profile")
orders_df = spark.read.table("dbt_adv_pr.silver.order_2025_all")

catalog_single_source = (
    sup_df
    .groupBy("sku_id")
    .agg(
        F.countDistinct("supplier_id").alias("registered_supplier_count"),
        F.first("supplier_id").alias("sole_supplier_id"),
        F.round(F.avg("unit_price"), 2).alias("catalog_unit_price"),
        F.round(F.avg("qc_rating"), 2).alias("qc_rating")
    )
    .filter(F.col("registered_supplier_count") == 1)
)

po_supplier_counts = (
    po_df
    .groupBy("sku_id")
    .agg(
        F.countDistinct("supplier_id").alias("active_po_suppliers"),
        F.sum("order_qty").alias("total_po_qty")
    )
)

sku_revenue = (
    orders_df
    .groupBy("sku_id")
    .agg(
        F.first("item_name").alias("item_name"),
        F.round(F.sum("amount"), 2).alias("total_revenue_at_risk"),
        F.sum("quantity").alias("total_units_sold")
    )
)

supply_risk_df = (
    catalog_single_source
    .join(po_supplier_counts, on="sku_id", how="left")
    .join(sku_revenue, on="sku_id", how="left")
    .join(
        profile_df.select(
            F.col("supplier_id").alias("sole_supplier_id"),
            F.col("name").alias("sole_supplier_name"),
            F.col("city").alias("supplier_city")
        ),
        on="sole_supplier_id",
        how="left"
    )
    .select(
        "sku_id",
        "item_name",
        "sole_supplier_id",
        "sole_supplier_name",
        "supplier_city",
        "registered_supplier_count",
        "active_po_suppliers",
        "qc_rating",
        "total_revenue_at_risk"
    )
    .orderBy(F.col("total_revenue_at_risk").desc())
)

supply_risk_df.show(30, truncate=False)


In [ ]:
supply_risk_df.write.mode("append")\
                    .saveAsTable("sku_supply_risk")

In [20]:
inv_df = spark.read.table("dbt_adv_pr.silver.inventory_2025_all")
late_pos = (
    po_df
    .filter(F.col("received_date").isNotNull() & (F.col("received_date") > F.col("expected_date")))
    .withColumn("days_late", F.datediff(F.col("received_date"), F.col("expected_date")))
    .select("po_id", "supplier_id", "warehouse_id", "sku_id", "expected_date", "received_date", "days_late")
)

stockouts_in_delay = (
    late_pos.alias("p")
    .join(
        inv_df.alias("i"),
        (F.col("p.warehouse_id") == F.col("i.warehouse_id")) &
        (F.col("p.sku_id") == F.col("i.sku_id")) &
        (F.col("i.snapshot_date") >= F.col("p.expected_date")) &
        (F.col("i.snapshot_date") < F.col("p.received_date")),
        how="inner"
    )
    .filter(F.col("i.on_hand_qty") == 0)
    .groupBy("p.po_id", "p.supplier_id", "p.warehouse_id", "p.sku_id", "p.days_late")
    .agg(F.countDistinct("i.snapshot_date").alias("stockout_days"))
)

date_span = orders_df.select(
    (F.datediff(F.max("order_date"), F.min("order_date")) + 1).alias("days")
).collect()[0]["days"]

sku_baseline = (
    orders_df
    .groupBy("warehouse_id", "sku_id")
    .agg(
        F.round(F.sum("quantity") / F.lit(date_span),2).alias("avg_daily_demand"),
        F.round(F.sum("amount") / F.sum("quantity"),2).alias("net_unit_price")
    )
)

late_delivery_cost_df = (
    stockouts_in_delay
    .join(sku_baseline, on=["warehouse_id", "sku_id"], how="left")
    .withColumn("estimated_lost_units", F.round(F.col("stockout_days") * F.col("avg_daily_demand"), 2))
    .withColumn("lost_sales_cost", F.round(F.col("estimated_lost_units") * F.col("net_unit_price"), 2))
    .orderBy(F.col("lost_sales_cost").desc())
)

late_delivery_cost_df.show(20, truncate=False)

(
    late_delivery_cost_df
    .groupBy("supplier_id")
    .agg(
        F.count("po_id").alias("late_pos_with_stockouts"),
        F.sum("stockout_days").alias("total_stockout_days"),
        F.round(F.sum("estimated_lost_units"), 0).alias("total_lost_units"),
        F.round(F.sum("lost_sales_cost"), 2).alias("total_lost_sales_cost")
    )
    .join(profile_df.select("supplier_id", "name"), on="supplier_id", how="left")
    .select("supplier_id", "name", "late_pos_with_stockouts", "total_stockout_days", "total_lost_units", "total_lost_sales_cost")
    .orderBy(F.col("total_lost_sales_cost").desc())
    .show(truncate=False)
)


+------------+------+--------+-----------+---------+-------------+----------------+--------------+--------------------+---------------+
|warehouse_id|sku_id|po_id   |supplier_id|days_late|stockout_days|avg_daily_demand|net_unit_price|estimated_lost_units|lost_sales_cost|
+------------+------+--------+-----------+---------+-------------+----------------+--------------+--------------------+---------------+
|WH02        |SKU002|PO000226|SUP03      |7        |7            |8.55            |297.54        |59.85               |17807.77       |
|WH01        |SKU012|PO000583|SUP02      |5        |4            |14.12           |250.03        |56.48               |14121.69       |
|WH03        |SKU001|PO000154|SUP06      |7        |4            |6.15            |562.86        |24.6                |13846.36       |
|WH02        |SKU001|PO000907|SUP02      |5        |3            |7.65            |564.58        |22.95               |12957.11       |
|WH01        |SKU003|PO000782|SUP07      |8     

In [21]:
late_delivery_cost_df.write.mode('append')\
                           .saveAsTable('dbt_adv_pr.gold.sku_late_delivery_cost') 